In [ ]:
#| default_exp renderers.seedream_4_5

# renderers.seedream_4_5

> Renderer for Seedream 4.5 (`bytedance/seedream-4.5`) via Replicate.
>
> Supports multi-image reference input for character consistency.
> No LoRA or negative prompt support.
> API key: `REPLICATE_API_TOKEN` environment variable.

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import os
from pathlib import Path

from manhualizer.config import OutputConfig, RendererConfig
from manhualizer.models import Panel, RenderResult
from manhualizer.render import BaseRenderer, ModelSpec

In [ ]:
#| export
_MODEL_ID = "bytedance/seedream-4.5"

class Seedream45Renderer(BaseRenderer):
    """Image generation via Seedream 4.5 (bytedance/seedream-4.5) on Replicate.

    Supports multi-image reference input (up to 14 images) for character consistency.
    Uses 'custom' size mode with dimensions from OutputConfig.

    Requires: REPLICATE_API_TOKEN environment variable.
    """

    def _client(self):
        import replicate  # type: ignore
        if not os.environ.get("REPLICATE_API_TOKEN"):
            raise EnvironmentError("REPLICATE_API_TOKEN is not set")
        return replicate

    def _build_input(
        self,
        panel: Panel,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> dict:
        w, h = output_cfg.resolved_dimensions()
        inp: dict = {
            "prompt": panel.visual_prompt,
            "size": "custom",
            "width": w,
            "height": h,
            "sequential_image_generation": "disabled",
        }
        if reference_images:
            # Replicate client accepts open file handles; it uploads them automatically.
            # Only include images for characters appearing in this panel.
            chars_in_panel = {c.lower() for c in (panel.characters or [])}
            images = [
                open(path, "rb")
                for name, path in reference_images.items()
                if name.lower() in chars_in_panel
            ]
            if images:
                inp["image_input"] = images
        return inp

    async def render_async(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        import asyncio
        return await asyncio.get_event_loop().run_in_executor(
            None, self._render_sync, panel, output_dir, output_cfg, reference_images
        )

    def _render_sync(
        self,
        panel: Panel,
        output_dir: Path,
        output_cfg: OutputConfig,
        reference_images: dict[str, Path] | None = None,
    ) -> RenderResult:
        import httpx
        replicate = self._client()

        inp = self._build_input(panel, output_cfg, reference_images)
        output = replicate.run(_MODEL_ID, input=inp)

        url = output[0] if isinstance(output[0], str) else output[0].url
        image_bytes = httpx.get(url).content

        out_path = output_dir / f"panel_{panel.panel_number:04d}.{output_cfg.format}"
        out_path.write_bytes(image_bytes)

        return RenderResult(
            panel_number=panel.panel_number,
            image_path=out_path,
            backend_used=self.model_spec.name,
            prompt_used=panel.visual_prompt,
            metadata={"ref_images": len(inp.get("image_input", []))},
        )

In [ ]:
# Construction test (no API call)
from manhualizer.render import MODELS
from manhualizer.config import RendererConfig
from manhualizer.renderers.seedream_4_5 import Seedream45Renderer

renderer = Seedream45Renderer(MODELS["seedream-4.5"], RendererConfig())
assert renderer.model_spec.capabilities.reference_images
assert renderer.model_spec.capabilities.multi_image_input
assert not renderer.model_spec.capabilities.lora
assert not renderer.model_spec.capabilities.negative_prompt
print("Seedream45Renderer OK")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()